# You are bot — итоговое решение

Задача: по диалогу определить, является ли каждый из двух участников ботом. Метрика — LogLoss по вероятностям.

**Решение — одна модель LightGBM.** Каждый участник диалога — одна строка. Признаки:
1. TF-IDF по буквенным сочетаниям его реплик и реплик собеседника;
2. простые статистики текста (длина, пунктуация, латиница, вопросы, повторы);
3. признаки из эмбеддингов замороженного `rubert-tiny2`: средний смысл реплик и то, насколько реплики отвечают по теме собеседнику.

Поверх модели — **температура**, которая исправляет самоуверенность вероятностей.

Качество на кросс-валидации (5 фолдов по диалогам): **LogLoss 0.195, с температурой 0.191**; ROC-AUC 0.977, F1 0.902 при пороге 0.5.

Запуск: данные в `data/` в корне проекта, затем «Run All». Работает на CPU: эмбеддинги считаются меньше минуты, вся кросс-валидация — несколько минут. В конце сохраняется `submissions/submission.csv`.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline

# ноутбук лежит в notebooks/, а пути к данным и кэшу — от корня проекта
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

SEED = 42
N_SPLITS = 5

## 1. Данные

`train.json` / `test.json` — словарь `{dialog_id: [сообщения]}`, у сообщения есть `text`, порядковый номер `message` и `participant_index` (0 или 1). Метки `is_bot` — в `ytrain.csv` для каждой пары (диалог, участник).

Разворачиваем диалоги в таблицу сообщений, затем собираем одну строку на участника: все его реплики одним текстом, число сообщений и средняя длина.

In [2]:
def load_messages(path):
    with open(path, encoding='utf-8') as f:
        dialogs = json.load(f)
    rows = [
        {'dialog_id': dialog_id, **msg}
        for dialog_id, messages in dialogs.items()
        for msg in messages
    ]
    return pd.DataFrame(rows)

train_msgs = load_messages('data/train.json')
test_msgs = load_messages('data/test.json')
print('Сообщений:', len(train_msgs), len(test_msgs))
train_msgs.head()

Сообщений: 10490 2837


,dialog_id,text,message,participant_index
0,b5b8e0ae-5516-4150-9004-00fe422d5e33,"how can i helpt you, men?",0,0
1,b5b8e0ae-5516-4150-9004-00fe422d5e33,Никак,1,1
2,b5b8e0ae-5516-4150-9004-00fe422d5e33,неа,2,0
3,b5b8e0ae-5516-4150-9004-00fe422d5e33,лох,3,1
4,b5b8e0ae-5516-4150-9004-00fe422d5e33,не тупи,4,0


In [3]:
def aggregate(msgs):
    msgs = msgs.sort_values(['dialog_id', 'message'])
    return (
        msgs.groupby(['dialog_id', 'participant_index'])
        .agg(
            text=('text', lambda s: ' \n '.join(s)),
            n_messages=('text', 'size'),
            mean_len=('text', lambda s: s.str.len().mean()),
        )
        .reset_index()
    )

ytrain = pd.read_csv('data/ytrain.csv')
ytest = pd.read_csv('data/ytest.csv')

# how='left' от файла с метками: участник без сообщений всё равно остаётся в выборке
train = ytrain.merge(aggregate(train_msgs), on=['dialog_id', 'participant_index'], how='left')
test = ytest.merge(aggregate(test_msgs), on=['dialog_id', 'participant_index'], how='left')
train = train.fillna({'text': '', 'n_messages': 0, 'mean_len': 0})
test = test.fillna({'text': '', 'n_messages': 0, 'mean_len': 0})

print('Участников:', train.shape[0], test.shape[0])
print('Доля ботов в train:', round(train['is_bot'].mean(), 3))
print('Ботов в диалоге:', train.groupby('dialog_id')['is_bot'].sum().value_counts().to_dict())

Участников: 3032 758
Доля ботов в train: 0.408
Ботов в диалоге: {1: 1237, 0: 279}


## 2. Признаки

### 2.1. Собеседник
В каждом диалоге ровно два участника, поэтому собеседник — строка того же диалога с `participant_index = 1 - participant_index`. Его реплики сильно помогают: человек, говорящий с ботом, часто его проверяет («ты бот?»), а бот отвечает развёрнуто и по теме.

In [6]:
def add_partner(df):
    partner = (
        df[['dialog_id', 'participant_index', 'text', 'n_messages', 'mean_len']]
        .assign(participant_index=lambda d: 1 - d['participant_index'])
        .rename(columns={'text': 'partner_text', 'n_messages': 'partner_n_messages', 'mean_len': 'partner_mean_len'})
    )
    return df.merge(partner, on=['dialog_id', 'participant_index'], how='left')

train = add_partner(train)
test = add_partner(test)

### 2.2. Статистики текста
Доли латиницы, заглавных и пунктуации, частота вопросов и скобочек-смайлов, длина в словах, самая длинная реплика и доля уникальных реплик. Текст специально не чистим: эмодзи, markdown, регистр и опечатки — это и есть сигнал «бот / человек».

In [7]:
stat_cols = ['latin', 'upper', 'punct', 'question', 'emoji', 'mean_words', 'max_len', 'unique_ratio']

def text_stats(msgs):
    s = msgs['text']
    length = s.str.len().clip(lower=1)
    return (
        msgs.assign(
            latin=s.str.count(r'[A-Za-z]') / length,
            upper=s.str.count(r'[A-ZА-ЯЁ]') / length,
            punct=s.str.count(r'[.,!?;:]') / length,
            question=s.str.contains(r'\?').astype(int),
            emoji=s.str.contains(r'[)(]').astype(int),
            n_words=s.str.split().str.len(),
        )
        .groupby(['dialog_id', 'participant_index'])
        .agg(
            latin=('latin', 'mean'),
            upper=('upper', 'mean'),
            punct=('punct', 'mean'),
            question=('question', 'mean'),
            emoji=('emoji', 'mean'),
            mean_words=('n_words', 'mean'),
            max_len=('text', lambda t: t.str.len().max()),
            unique_ratio=('text', lambda t: t.nunique() / len(t)),
        )
        .reset_index()
    )

def add_stats(df, msgs):
    stats = text_stats(msgs)
    partner = (
        stats.assign(participant_index=lambda d: 1 - d['participant_index'])
        .rename(columns={c: 'partner_' + c for c in stat_cols})
    )
    return (
        df.merge(stats, on=['dialog_id', 'participant_index'], how='left')
        .merge(partner, on=['dialog_id', 'participant_index'], how='left')
    )

train = add_stats(train, train_msgs)
test = add_stats(test, test_msgs)
train.groupby('is_bot')[stat_cols].mean().T

is_bot,0,1
latin,0.230058,0.188143
upper,0.077150,0.048325
punct,0.038136,0.036796
question,0.198912,0.290288
emoji,0.023555,0.047818
mean_words,2.552303,5.284037
max_len,75.994986,63.772029
unique_ratio,0.963231,0.952789


### 2.3. Эмбеддинги rubert-tiny2
Каждое сообщение кодируем **замороженным** `cointegrated/rubert-tiny2` (29M параметров, вектор из 312 чисел) — модель не дообучается. Векторы нормированы, поэтому скалярное произведение — косинусная близость. Признаки участника:
- `coh_mean / coh_min / coh_max` — близость каждой его реплики к предыдущей реплике собеседника: отвечает ли он по теме;
- `self_sim_mean` — близость соседних собственных реплик: повторяется ли он;
- `emb_0 … emb_311` — средний эмбеддинг его реплик.

Те же признаки собеседника — с префиксом `partner_`. Если перед репликами участника не было реплик собеседника, `coh_*` пустые — LightGBM умеет с этим работать.

Эмбеддинги считает `scripts/embed.py` отдельным процессом и сохраняет в `cache/`: на macOS torch и lightgbm в одном процессе роняют ядро. Векторы считаются один раз; чтобы пересчитать, удалите `cache/`.

In [8]:
if not (Path('cache/emb_train.npy').exists() and Path('cache/emb_test.npy').exists()):
    print(subprocess.run([sys.executable, 'scripts/embed.py'], check=True, capture_output=True, text=True).stdout)

train_emb = np.load('cache/emb_train.npy')
test_emb = np.load('cache/emb_test.npy')
assert len(train_emb) == len(train_msgs) and len(test_emb) == len(test_msgs)
train_emb.shape, test_emb.shape

((10490, 312), (2837, 312))

In [9]:
def embedding_features(msgs, emb):
    rows = []
    for dialog_id, d in msgs.reset_index(drop=True).groupby('dialog_id', sort=False):
        d = d.sort_values('message')
        who = d['participant_index'].to_numpy()
        e = emb[d.index.to_numpy()]
        for p in (0, 1):
            own = np.flatnonzero(who == p)
            coh, self_sim = [], []
            for k, i in enumerate(own):
                prev_partner = np.flatnonzero(who[:i] != p)
                if len(prev_partner):
                    coh.append(e[i] @ e[prev_partner[-1]])
                if k > 0:
                    self_sim.append(e[i] @ e[own[k - 1]])
            mean_emb = e[own].mean(axis=0) if len(own) else np.zeros(emb.shape[1])
            rows.append({
                'dialog_id': dialog_id,
                'participant_index': p,
                'coh_mean': np.mean(coh) if coh else np.nan,
                'coh_min': np.min(coh) if coh else np.nan,
                'coh_max': np.max(coh) if coh else np.nan,
                'self_sim_mean': np.mean(self_sim) if self_sim else np.nan,
                **{f'emb_{j}': v for j, v in enumerate(mean_emb)},
            })
    return pd.DataFrame(rows)

def add_embedding_features(df, msgs, emb):
    feats = embedding_features(msgs, emb)
    cols = [c for c in feats.columns if c not in ('dialog_id', 'participant_index')]
    partner = (
        feats.assign(participant_index=lambda d: 1 - d['participant_index'])
        .rename(columns={c: 'partner_' + c for c in cols})
    )
    return (
        df.merge(feats, on=['dialog_id', 'participant_index'], how='left')
        .merge(partner, on=['dialog_id', 'participant_index'], how='left')
    )

train = add_embedding_features(train, train_msgs, train_emb)
test = add_embedding_features(test, test_msgs, test_emb)

coh_cols = ['coh_mean', 'coh_min', 'coh_max', 'self_sim_mean']
emb_cols = [c for c in train.columns if c.startswith('emb_')]
train.groupby('is_bot')[coh_cols + ['partner_' + c for c in coh_cols]].mean().T

is_bot,0,1
coh_mean,0.717453,0.806020
coh_min,0.646875,0.741653
coh_max,0.785687,0.864868
self_sim_mean,0.751984,0.750168
partner_coh_mean,0.779386,0.717054
partner_coh_min,0.716514,0.641407
partner_coh_max,0.837949,0.789990
partner_self_sim_mean,0.740893,0.767279


## 3. Модель

LightGBM на всех признаках сразу. TF-IDF строится внутри пайплайна, поэтому на кросс-валидации словарь собирается только по обучающим фолдам. Гиперпараметры подобраны Optuna (60 итераций, код — в `eda.ipynb`).

**Валидация** — `GroupKFold` по `dialog_id`: оба участника одного диалога всегда в одном фолде, иначе модель видела бы собеседника на обучении и оценка была бы завышена. `cross_val_predict` даёт для каждого участника train предсказание модели, которая его не видела (OOF).

In [10]:
numeric_features = (
    ['n_messages', 'mean_len', 'partner_n_messages', 'partner_mean_len']
    + stat_cols + ['partner_' + c for c in stat_cols]
    + coh_cols + ['partner_' + c for c in coh_cols]
    + emb_cols + ['partner_' + c for c in emb_cols]
)
features = ['text', 'partner_text'] + numeric_features
print('Числовых признаков:', len(numeric_features), '+ TF-IDF до 2 × 20 000')

model = Pipeline([
    ('features', ColumnTransformer([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=3, max_features=20000, sublinear_tf=True), 'text'),
        ('partner_char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=3, max_features=20000, sublinear_tf=True), 'partner_text'),
        ('num', 'passthrough', numeric_features),
    ])),
    ('clf', lgb.LGBMClassifier(
        n_estimators=700, learning_rate=0.024, num_leaves=22, min_child_samples=18,
        subsample=0.94, subsample_freq=1, colsample_bytree=0.5, reg_lambda=0.53,
        random_state=SEED, verbose=-1,
    )),
])

cv = GroupKFold(n_splits=N_SPLITS)
y = train['is_bot'].to_numpy()

oof = cross_val_predict(
    model, train[features], y,
    groups=train['dialog_id'], cv=cv, method='predict_proba',
)[:, 1]
print('CV LogLoss:', round(log_loss(y, oof), 4))
print('Константа (доля ботов):', round(log_loss(y, np.full(len(y), y.mean())), 4))

Числовых признаков: 652 + TF-IDF до 2 × 20 000
CV LogLoss: 0.1954
Константа (доля ботов): 0.6761


## 4. Температура

LightGBM самоуверенна: тем, кого она считает почти наверняка людьми, на деле боты чаще, чем она говорит. Температура смягчает вероятности, не меняя их порядок:

$$p' = \sigma\left(\frac{\operatorname{logit}(p)}{T}\right)$$

При `T > 1` вероятности сдвигаются к 0.5. `T` подбираем по OOF-предсказаниям. Для честной оценки — вложенно: подбираем на четырёх фолдах и применяем к пятому.

In [11]:
def apply_temperature(p, T):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return 1 / (1 + np.exp(-np.log(p / (1 - p)) / T))

def fit_temperature(p, target):
    return minimize_scalar(lambda T: log_loss(target, apply_temperature(p, T)), bounds=(0.3, 3), method='bounded').x

TEMPERATURE = fit_temperature(oof, y)

calibrated_oof = np.zeros(len(y))
for tr, va in cv.split(train, y, groups=train['dialog_id']):
    calibrated_oof[va] = apply_temperature(oof[va], fit_temperature(oof[tr], y[tr]))

print(f'Температура: {TEMPERATURE:.3f}')
print(f'CV LogLoss: без температуры {log_loss(y, oof):.4f}, с температурой {log_loss(y, calibrated_oof):.4f}')

Температура: 1.242
CV LogLoss: без температуры 0.1954, с температурой 0.1908


In [12]:
bins = pd.cut(oof, np.linspace(0, 1, 11), include_lowest=True)
pd.DataFrame({
    'участников': pd.Series(y).groupby(bins, observed=False).size(),
    'предсказано': pd.Series(oof).groupby(bins, observed=False).mean().round(3),
    'после температуры': pd.Series(apply_temperature(oof, TEMPERATURE)).groupby(bins, observed=False).mean().round(3),
    'ботов на деле': pd.Series(y).groupby(bins, observed=False).mean().round(3),
})

,участников,предсказано,после температуры,ботов на деле
"(-0.001, 0.1]",1560,0.013,0.024,0.029
"(0.1, 0.2]",123,0.144,0.192,0.211
"(0.2, 0.3]",68,0.251,0.293,0.309
"(0.3, 0.4]",77,0.350,0.378,0.403
"(0.4, 0.5]",43,0.449,0.458,0.721
"(0.5, 0.6]",38,0.553,0.543,0.500
"(0.6, 0.7]",42,0.654,0.626,0.690
"(0.7, 0.8]",58,0.749,0.707,0.862
"(0.8, 0.9]",104,0.851,0.804,0.837
"(0.9, 1.0]",919,0.984,0.971,0.976


## 5. Метрики классификации

Для справки — метрики по меткам при разных порогах. Лидерборд считается по LogLoss, порог на сабмит не влияет. Температура не меняет порядок участников и переводит 0.5 в 0.5, поэтому F1 при пороге 0.5 с ней такой же.

In [13]:
print('ROC-AUC:', round(roc_auc_score(y, oof), 4))
pd.DataFrame([
    {
        'порог': t,
        'F1': f1_score(y, oof >= t),
        'precision': precision_score(y, oof >= t),
        'recall': recall_score(y, oof >= t),
    }
    for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
]).round(3)

ROC-AUC: 0.9772


,порог,F1,precision,recall
0,0.2,0.901,0.864,0.942
1,0.3,0.909,0.893,0.925
2,0.4,0.912,0.924,0.900
3,0.5,0.902,0.932,0.875
4,0.6,0.901,0.947,0.859
5,0.7,0.892,0.957,0.836
6,0.8,0.871,0.962,0.795


## 6. Сабмит

Обучаем модель на всём train, предсказываем test и применяем температуру. После температуры обрезка нужна только от вероятностей ровно 0 и 1 — они дали бы бесконечный LogLoss.

In [14]:
CLIP = 0.001

model.fit(train[features], y)
preds = apply_temperature(model.predict_proba(test[features])[:, 1], TEMPERATURE)

submission = test[['ID']].assign(is_bot=np.clip(preds, CLIP, 1 - CLIP))
assert len(submission) == len(ytest)
assert (submission['ID'] == ytest['ID']).all()
submission.to_csv('submissions/submission.csv', index=False)
submission.head()

,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.028077
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.049631
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.975725
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.021425
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.093100
